# Boundary outlines

Turn a 2D shape into closed loops of boundary vertices, at a resolution you choose. Boundary-element and panel solvers discretise electrodes and walls from these loops; they also give perimeters, areas and plot outlines.

`shape.outlines(max_spacing=None)` returns a list of `(2, n)` NumPy arrays, one per piece of the shape. Each loop is counter-clockwise and does not repeat its first vertex.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from simgeometry import Aperture2D, Circle, Polygon, Rectangle

rectangle = Rectangle(midpoint=(0.0, 0.0), side_lengths=(2.0, 1.0))
(loop,) = rectangle.outlines()
print(loop)
assert loop.shape == (2, 4)

Without `max_spacing`, a polygon's outline is its corners and a circle gets 100 vertices. A shape made of several pieces has one loop per piece: an `Aperture2D` has an upper and a lower plate. Inverting a shape does not change its outline.

In [ ]:
slit = Aperture2D(
    axis=0,
    position=0.0,
    vertices=((-0.25, 0.3), (0.25, 0.3), (0.25, 1.2), (-0.25, 1.2)),
)
shapes = {
    "rectangle": rectangle,
    "circle": Circle(midpoint=(0.0, 0.0), radius=0.6),
    "L-shaped polygon": Polygon(
        vertices=np.array([(0, 0), (2, 0), (2, 1), (1, 1), (1, 2), (0, 2)], dtype=float)
    ),
    "slit aperture": slit,
}

fig, axes = plt.subplots(1, 4, figsize=(12, 3), layout="constrained")
for ax, (name, shape) in zip(axes, shapes.items()):
    for loop in shape.outlines():
        closed = np.concatenate((loop, loop[:, :1]), axis=1)
        ax.plot(*closed, "-o", markersize=3)
    ax.set_title(f"{name}: {len(shape.outlines())} loop(s)")
    ax.set_aspect("equal")
axes[-1].set_xticks([-0.25, 0.25])
plt.show()

assert len(slit.outlines()) == 2
inverted = Circle(midpoint=(0.0, 0.0), radius=0.6, inverted=True)
np.testing.assert_array_equal(inverted.outlines()[0], shapes["circle"].outlines()[0])

## Choosing the spacing

With a number for `max_spacing`, every edge longer than it is split into equal pieces, so no two consecutive vertices are further apart. Polygon corners stay exact. On a circle every vertex lies on the circle, so the loop is an inscribed polygon whose perimeter and area converge as the spacing shrinks.

In [ ]:
def perimeter(loop):
    closed = np.concatenate((loop, loop[:, :1]), axis=1)
    return np.hypot(*np.diff(closed, axis=1)).sum()


def area(loop):
    x, y = loop
    return 0.5 * np.sum(x * np.roll(y, -1) - np.roll(x, -1) * y)


circle = Circle(midpoint=(0.0, 0.0), radius=1.0)
spacings = [0.8, 0.4, 0.2, 0.1, 0.05, 0.025]
perimeter_error, area_error = [], []
for spacing in spacings:
    (loop,) = circle.outlines(max_spacing=spacing)
    closed = np.concatenate((loop, loop[:, :1]), axis=1)
    assert np.hypot(*np.diff(closed, axis=1)).max() <= spacing + 1e-12
    np.testing.assert_allclose(np.hypot(*loop), 1.0)
    perimeter_error.append(2 * np.pi - perimeter(loop))
    area_error.append(np.pi - area(loop))
    print(f"max_spacing {spacing:5.3f}: {loop.shape[1]:4d} vertices")

fig, ax = plt.subplots(figsize=(5, 3.5), layout="constrained")
ax.loglog(spacings, perimeter_error, "o-", label="perimeter error")
ax.loglog(spacings, area_error, "s-", label="area error")
ax.loglog(spacings, 0.2 * np.square(spacings), "k--", label="spacing²")
ax.set_xticks(spacings, [f"{spacing:g}" for spacing in spacings], minor=False)
ax.set_xticks([], minor=True)
ax.set_xlabel("max_spacing")
ax.set_ylabel("error of the inscribed polygon")
ax.legend()
plt.show()

(l_shape,) = shapes["L-shaped polygon"].outlines(max_spacing=0.25)
assert area(l_shape) == 3.0  # refining a polygon only adds points on its edges

Both errors fall as the square of the spacing, the accuracy of a polygon inscribed in a smooth curve.

## Graded spacing

`max_spacing` can also be a function: it takes `(2, m)` points and returns `(m,)` spacings. Vertices are then placed at about the local spacing, so you can resolve a corner or an aperture tip finely and keep the rest coarse. A boundary-element solver needs short panels where the field varies fastest, and a slit's field varies fastest at its tip. With a function, the spacing holds approximately, not as a strict bound.

In [ ]:
def tip_spacing(points):
    """0.02 at the slit tips (|y| = 0.3), growing with distance, at most 0.2."""
    distance = np.abs(points[1]) - 0.3
    return np.minimum(0.02 + 0.25 * distance, 0.2)


uniform = slit.outlines(max_spacing=0.02)
graded = slit.outlines(max_spacing=tip_spacing)
n_uniform = sum(loop.shape[1] for loop in uniform)
n_graded = sum(loop.shape[1] for loop in graded)
print("vertices, uniform 0.02:", n_uniform)
print("vertices, graded:      ", n_graded)

fig, (ax0, ax1) = plt.subplots(
    1, 2, figsize=(10, 3.5), layout="constrained", width_ratios=(1, 2)
)
for loop in graded:
    closed = np.concatenate((loop, loop[:, :1]), axis=1)
    ax0.plot(*closed, "-o", markersize=2)
ax0.set_aspect("equal")
ax0.set_xticks([-0.25, 0.25])
ax0.set_title("Graded vertices on the slit")

upper = graded[1]
closed = np.concatenate((upper, upper[:, :1]), axis=1)
midpoints = 0.5 * (closed[:, 1:] + closed[:, :-1])
edges = np.hypot(*np.diff(closed, axis=1))
order = np.argsort(midpoints[1])
ax1.plot(midpoints[1][order], edges[order], ".", label="edge length")
y = np.linspace(0.3, 1.2, 200)
ax1.plot(y, tip_spacing(np.stack((np.zeros_like(y), y))), "k-", label="requested")
ax1.set_xlabel("y (distance along the plate)")
ax1.set_ylabel("spacing")
ax1.legend()
plt.show()

assert edges[midpoints[1] < 0.35].max() < 0.03
assert n_graded < n_uniform / 2

The graded outline matches the uniform one at the tip with about a third of the vertices. Edges come out at or a little below the requested spacing, because each edge of the shape is split into a whole number of pieces.

## Transformed shapes

Shapes from `translated()`, `rotated()`, `scaled()` and `transformed()` have outlines too: the source's loops, mapped by the transform. The spacing applies after the transform, both as a number and as a function of world coordinates. Boolean combinations (`a | b`, `a - b`, ...) have no outline; their boundary is not available as loops.

In [ ]:
bar = Rectangle(midpoint=(0.0, 0.0), side_lengths=(2.0, 0.4))
tilted = bar.rotated(30).scaled((1.0, 2.0)).translated((1.0, 0.5))
(loop,) = tilted.outlines(max_spacing=0.1)
closed = np.concatenate((loop, loop[:, :1]), axis=1)
assert np.hypot(*np.diff(closed, axis=1)).max() <= 0.1 + 1e-12
assert area(loop) > 0  # still counter-clockwise

(source,) = bar.outlines()
fig, ax = plt.subplots(figsize=(5, 3.5), layout="constrained")
ax.plot(*np.concatenate((source, source[:, :1]), axis=1), "k--", label="source")
ax.plot(*closed, "-o", markersize=2, label="rotated, scaled, translated")
ax.set_aspect("equal")
ax.legend()
plt.show()

Try a graded spacing that refines the corners of the L-shaped polygon, where the field of a charged conductor is singular.